In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [2]:
#remove 2 suspecious outliers
train = train[~((train['GrLivArea'] > 4000) & 
                (train['SalePrice'] < 200000))]
print('Train shape after removing outliers:', train.shape)

Train shape after removing outliers: (1458, 81)


In [3]:
#separate target variable befor transformation

y_train = np.log1p(train['SalePrice'])

train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop(['Id'], axis=1)

print('Train features shape:', train.shape)
print('Test shape:', test.shape)
print('Target shape:', y_train.shape)
print('\nFirst 5 target values (log transformed):')
print(y_train.head())

Train features shape: (1458, 79)
Test shape: (1459, 79)
Target shape: (1458,)

First 5 target values (log transformed):
0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64


In [20]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 
             'FireplaceQu', 'GarageType', 'GarageFinish',
             'GarageQual', 'GarageCond', 'BsmtQual', 
             'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
             'BsmtFinType2', 'MasVnrType']

for col in none_cols:
    train[col] = train[col].fillna('None')
    test[col]  = test[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageCars', 'GarageArea',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
             'TotalBsmtSF', 'MasVnrArea', 'BsmtFullBath', 
             'BsmtHalfBath']

for col in zero_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

lot_medians = train.groupby('Neighborhood')['LotFrontage'].median()
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
test['LotFrontage'] = test['Neighborhood'].map(lot_medians).where(test['LotFrontage'].isna(), test['LotFrontage'])

In [ ]:
# Electrical → 1 missing, fill with mode from TRAIN
electrical_mode = train['Electrical'].mode()[0]
train['Electrical'] = train['Electrical'].fillna(electrical_mode)
test['Electrical']  = test['Electrical'].fillna(electrical_mode)

# Row 332 and 948 specific fixes 
bsmt_fintype2_mode = train['BsmtFinType2'].mode()[0]
bsmt_exposure_mode = train['BsmtExposure'].mode()[0]

train['BsmtFinType2'] = train['BsmtFinType2'].fillna(bsmt_fintype2_mode)
test['BsmtFinType2']  = test['BsmtFinType2'].fillna(bsmt_fintype2_mode)

train['BsmtExposure'] = train['BsmtExposure'].fillna(bsmt_exposure_mode)
test['BsmtExposure']  = test['BsmtExposure'].fillna(bsmt_exposure_mode)

In [21]:
# Verify: no missing values remaining
train_missing = train.isnull().sum().sum()
test_missing  = test.isnull().sum().sum()

print(f'Train missing values remaining: {train_missing}')
print(f'Test missing values remaining:  {test_missing}')

Train missing values remaining: 1
Test missing values remaining:  12


In [22]:
# find which column still has missing value in train
print("Train missing columns:")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\nTest missing columns:")
print(test.isnull().sum()[test.isnull().sum() > 0])

Train missing columns:
Electrical    1
dtype: int64

Test missing columns:
MSZoning       4
Utilities      2
Exterior1st    1
Exterior2nd    1
KitchenQual    1
Functional     2
SaleType       1
dtype: int64


In [23]:
# columns that need mode filling
mode_cols = ['Electrical', 'MSZoning', 'Utilities', 
             'Exterior1st', 'Exterior2nd', 'KitchenQual', 
             'Functional', 'SaleType']

for col in mode_cols:
    mode_val = train[col].mode()[0]
    
    train[col] = train[col].fillna(mode_val)
    test[col]  = test[col].fillna(mode_val)

# verify
print(f'Train missing: {train.isnull().sum().sum()}')
print(f'Test missing:  {test.isnull().sum().sum()}')

Train missing: 0
Test missing:  0
